# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [47]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, f1_score, precision_score, recall_score

In [32]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [33]:
df = spaceship.dropna()
df.shape

(6606, 14)

In [34]:
df.loc[:, "Cabin"] = df["Cabin"].str[0]
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [35]:
df = df.drop(columns=["PassengerId", "Name"])

In [36]:
X = df.drop("Transported", axis=1)
y = df["Transported"]

nnum_cols = X.select_dtypes(include=["object"]).columns
X = pd.get_dummies(X, columns=nnum_cols, drop_first=True)

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

**Perform Train Test Split**

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

- Train the model and make predictions

In [44]:
xgb_basic = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
)

xgb_basic.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

- Evaluate your model

In [45]:
xgb_basic_accuracy = xgb_basic.score(X_test, y_test)
print("XGBoost Basic Accuracy:", xgb_basic_accuracy)

y_pred_basic = xgb_basic.predict(X_test)
f1_basic = f1_score(y_test, y_pred_basic)
print("F1 Score:", f1_basic)

y_proba_basic = xgb_basic.predict_proba(X_test)[:, 1]

print("\nTest ROC-AUC:", roc_auc_score(y_test, y_proba_basic))
print("Test Accuracy:", accuracy_score(y_test, y_pred_basic))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_basic))

XGBoost Basic Accuracy: 0.8184568835098336
F1 Score: 0.8237885462555066

Test ROC-AUC: 0.88823116031104
Test Accuracy: 0.8184568835098336

Classification Report:
              precision    recall  f1-score   support

       False       0.83      0.80      0.81       653
        True       0.81      0.84      0.82       669

    accuracy                           0.82      1322
   macro avg       0.82      0.82      0.82      1322
weighted avg       0.82      0.82      0.82      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [40]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=3,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, ...)

- Run Grid Search

In [41]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=3,
    random_state=42,
    tree_method="hist"
)

param_grid = {
    "max_depth": [3, 4, 5],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.5, 0.8],
    "reg_lambda": [1, 3, 5]
}

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
[CV] END colsample_bytree=0.5, max_depth=3, min_child_weight=1, reg_lambda=1, subsample=0.7; total time=   0.9s
[CV] END colsample_bytree=0.5, max_depth=3, min_child_weight=1, reg_lambda=1, subsample=0.7; total time=   1.0s
[CV] END colsample_bytree=0.5, max_depth=3, min_child_weight=1, reg_lambda=1, subsample=0.7; total time=   1.0s
[CV] END colsample_bytree=0.5, max_depth=3, min_child_weight=1, reg_lambda=1, subsample=0.8; total time=   1.0s
[CV] END colsample_bytree=0.5, max_depth=3, min_child_weight=1, reg_lambda=1, subsample=0.7; total time=   1.0s
[CV] END colsample_bytree=0.5, max_depth=3, min_child_weight=1, reg_lambda=1, subsample=0.8; total time=   1.0s
[CV] END colsample_bytree=0.5, max_depth=3, min_child_weight=1, reg_lambda=1, subsample=0.7; total time=   1.2s
[CV] END colsample_bytree=0.5, max_depth=3, min_child_weight=1, reg_lambda=1, subsample=0.8; total time=   1.0s
[CV] END colsample_bytree=0.5, max_depth=

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=0.8, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='logloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_p...
                                     max_delta_step=None, max_depth=4,
                                     max_leaves=None, min_child_weight=3,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=1000,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.5, 0.8], 'max_depth': [3, 4, 5],
                         'min_child_weight': [1, 3, 5], 'reg_lambda': [1, 3, 5],
                         'subsample': [0.7, 0.8, 0.9]},
             scoring='roc_auc', verbose=2)

- Evaluate your model

In [42]:
print("Best parameters:")
print(grid.best_params_)

print("\nBest CV ROC-AUC:")
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("\nTest ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Best parameters:
{'colsample_bytree': 0.8, 'max_depth': 3, 'min_child_weight': 1, 'reg_lambda': 5, 'subsample': 0.8}

Best CV ROC-AUC:
0.8842774371362765

Test ROC-AUC: 0.8874368500447516
Test Accuracy: 0.8063540090771558

Classification Report:
              precision    recall  f1-score   support

       False       0.82      0.78      0.80       653
        True       0.80      0.83      0.81       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



Exception ignored in: <function ResourceTracker.__del__ at 0x104a75bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x104f49bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1078b9bc0>
Traceback (most recent call last

- Model Evaluation Comparison

In [49]:
# Create comparison table
results = pd.DataFrame({
    "Accuracy": [
        accuracy_score(y_test, y_pred_basic),
        accuracy_score(y_test, y_pred)
    ],
    "F1": [
        f1_score(y_test, y_pred_basic),
        f1_score(y_test, y_pred)
    ],
    "Precision": [
        precision_score(y_test, y_pred_basic),
        precision_score(y_test, y_pred)
    ],
    "Recall": [
        recall_score(y_test, y_pred_basic),
        recall_score(y_test, y_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_basic),
        roc_auc_score(y_test, y_proba)
    ]
}, index=["Basic XGBoost", "GridSearch XGBoost"])

# Round values for readability
results = results.round(4)

print("\nModel Comparison\n")
print(results)


Model Comparison

                    Accuracy      F1  Precision  Recall  ROC-AUC
Basic XGBoost         0.8185  0.8238     0.8095  0.8386   0.8882
GridSearch XGBoost    0.8064  0.8129     0.7954  0.8311   0.8874


- The basic model performs better than the tuned model